# Exercise 2 — Code Generation with ReACT Prompting

**Goal:** generate Python code using a Reason → Act → Observe loop: the model reasons and plans, writes code, the notebook **executes it against tests**, and the result is fed back as an observation until the tests pass.

**Tools:** Google Colab · Anthropic Python SDK (Claude) · Python `exec` sandbox + a small test harness

**Loop**

```
THOUGHT (reason)  →  PLAN  →  ACTION: write_code  →  [notebook runs tests]  →  OBSERVATION  →  (fix)  →  …  →  ACTION: done
```

## Setup

Uses the Anthropic Python SDK. Add your key to Colab **Secrets** (🔑 icon in the left sidebar) as `ANTHROPIC_API_KEY`, or paste it when prompted.

In [1]:
!pip install -q anthropic

import os, json, re, textwrap
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

client = Anthropic()
MODEL = "claude-sonnet-5"   # any current Claude model works; "claude-haiku-4-5" is cheaper/faster

def get_text(resp) -> str:
    """Join the text blocks of a response (newer models may also return thinking blocks)."""
    return "".join(b.text for b in resp.content if b.type == "text").strip()

def ask(system: str, user: str, max_tokens: int = 4000) -> str:
    """Single-turn call: system prompt + one user message -> text."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
        thinking={"type": "disabled"},   # the prompts themselves carry the reasoning steps
    )
    return get_text(resp)

def parse_json(text: str):
    """Tolerant JSON parse: strips ```json fences if the model adds them."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    return json.loads(cleaned)

def show(title: str, body: str):
    print(f"\n{'='*70}\n{title}\n{'='*70}\n{body}")

print("Ready. Model:", MODEL)

Ready. Model: claude-sonnet-5


## The task the model must solve
Deliberately includes edge cases (quoted commas, `$1,250.00`, malformed rows, empty input) so a naive `split(",")` solution fails and the loop has to fix it.

In [2]:
TASK = """Implement `summarize_orders(csv_text: str) -> dict` in pure Python (standard library only).

INPUT: a CSV string with header exactly: order_id,customer,amount,status
OUTPUT: {"revenue_by_customer": {customer: float}, "total_revenue": float, "skipped_rows": int}

REQUIREMENTS
1. Count only rows whose status is "paid" (case-insensitive, whitespace-trimmed).
2. amount may have a leading "$" and thousands separators, e.g. "$1,250.00". Parse to float; round every output float to 2 dp.
3. Customer names may be quoted and contain commas, e.g. "Smith, John".
4. Malformed rows (wrong field count, non-numeric amount, empty customer) are skipped AND counted in skipped_rows. Never raise on a bad row.
5. Empty input or header-only input returns {"revenue_by_customer": {}, "total_revenue": 0.0, "skipped_rows": 0}.
6. No printing, no input(), no third-party libraries.
"""
print(TASK)

Implement `summarize_orders(csv_text: str) -> dict` in pure Python (standard library only).

INPUT: a CSV string with header exactly: order_id,customer,amount,status
OUTPUT: {"revenue_by_customer": {customer: float}, "total_revenue": float, "skipped_rows": int}

REQUIREMENTS
1. Count only rows whose status is "paid" (case-insensitive, whitespace-trimmed).
2. amount may have a leading "$" and thousands separators, e.g. "$1,250.00". Parse to float; round every output float to 2 dp.
3. Customer names may be quoted and contain commas, e.g. "Smith, John".
4. Malformed rows (wrong field count, non-numeric amount, empty customer) are skipped AND counted in skipped_rows. Never raise on a bad row.
5. Empty input or header-only input returns {"revenue_by_customer": {}, "total_revenue": 0.0, "skipped_rows": 0}.
6. No printing, no input(), no third-party libraries.



## Test harness (the "environment" that produces observations)

In [3]:
import traceback

SAMPLE = '''order_id,customer,amount,status
1,Alice,$100.00,paid
2,"Smith, John","$1,250.00",PAID
3,Alice,50,refunded
4,Bob,not_a_number,paid
5,,20.00,paid
6,Bob," 30.5 ",paid
7,Bob,10,paid,extra_field
'''

TESTS = [
    ("basic + status filter",  SAMPLE,  lambda r: r["revenue_by_customer"].get("Alice") == 100.0),
    ("quoted name with comma", SAMPLE,  lambda r: r["revenue_by_customer"].get("Smith, John") == 1250.0),
    ("dollar sign + thousands",SAMPLE,  lambda r: r["total_revenue"] == 1380.5),
    ("skips malformed rows",   SAMPLE,  lambda r: r["skipped_rows"] == 3),
    ("empty input",            "",      lambda r: r == {"revenue_by_customer": {}, "total_revenue": 0.0, "skipped_rows": 0}),
    ("header only",            "order_id,customer,amount,status\n",
                                        lambda r: r["total_revenue"] == 0.0 and r["skipped_rows"] == 0),
]

def run_tests(code: str) -> tuple[bool, str]:
    """Execute the model's code, run every test, return (all_passed, observation_text)."""
    ns = {}
    try:
        exec(code, ns)
    except Exception:
        return False, "Code failed to execute:\n" + traceback.format_exc(limit=2)
    fn = ns.get("summarize_orders")
    if not callable(fn):
        return False, "No function named summarize_orders was defined."
    lines, passed = [], 0
    for name, inp, check in TESTS:
        try:
            out = fn(inp)
            ok = bool(check(out))
        except Exception as e:
            ok, out = False, f"raised {type(e).__name__}: {e}"
        passed += ok
        lines.append(f"[{'PASS' if ok else 'FAIL'}] {name}" + ("" if ok else f" -> got: {out!r}"))
    return passed == len(TESTS), f"{passed}/{len(TESTS)} tests passed\n" + "\n".join(lines)

## The ReACT prompt
Explicit stages, strict format, and rules about how to react to an observation (fix the specific failure, don't guess the tests).

In [4]:
REACT_SYSTEM = """You are a senior Python engineer working inside a ReAct loop (Reason -> Act -> Observe).
Every reply MUST use exactly this structure and nothing else:

THOUGHT: Reason about the task and constraints. If there is a previous OBSERVATION, state precisely
         what failed and why, referring to the specific requirement it violates.
PLAN: A short numbered list of the concrete steps your code will take (including edge-case handling).
ACTION: write_code
```python
<complete, self-contained code that defines summarize_orders. No test code, no print statements.>
```

Rules:
- Standard library only (csv, re, decimal, etc. are fine). No input(), no file I/O.
- Your code block is executed verbatim against hidden tests; the result comes back as an OBSERVATION.
- If the OBSERVATION says all tests passed, reply with:
  THOUGHT: <one sentence>
  ACTION: done
- If tests failed, fix the specific failure. Do not rewrite unrelated parts. Do not guess the test data.
- Handle every requirement in the task; requirement 4 means bad rows must never raise."""

def extract_code(reply: str) -> str | None:
    m = re.search(r"```python\s*(.*?)```", reply, re.S)
    return m.group(1).strip() if m else None

## Run the loop

In [5]:
def react_loop(task: str, max_iters: int = 4):
    messages = [{"role": "user", "content": f"TASK:\n{task}\n\nBegin."}]
    final_code = None
    for i in range(1, max_iters + 1):
        resp = client.messages.create(model=MODEL, max_tokens=8000, system=REACT_SYSTEM,
                                      messages=messages, thinking={"type": "disabled"})
        reply = get_text(resp)
        messages.append({"role": "assistant", "content": reply})
        show(f"ITERATION {i} · model (THOUGHT / PLAN / ACTION)", reply)

        if "ACTION: done" in reply:
            print("\nModel signalled completion.")
            break
        code = extract_code(reply)
        if code is None:
            obs = "OBSERVATION: no ```python code block found. Follow the required format."
        else:
            passed, report = run_tests(code)
            final_code = code
            obs = f"OBSERVATION:\n{report}"
        show(f"ITERATION {i} · environment", obs)
        messages.append({"role": "user", "content": obs})
    return final_code

FINAL_CODE = react_loop(TASK)


ITERATION 1 · model (THOUGHT / PLAN / ACTION)
THOUGHT: Need to parse CSV with proper quoting support (use csv module), handle amount cleaning, validate rows, and aggregate revenue per customer for paid orders only.

PLAN:
1. Use csv.reader with io.StringIO to handle quoted fields with commas.
2. Read header row; if no data rows, return default zero result.
3. For each row: check field count == 4; extract order_id, customer, amount, status.
4. Trim/validate customer (non-empty after strip), status must equal "paid" case-insensitive after strip to count as revenue; otherwise skip counting but still not necessarily an error—only rows failing validation count as skipped.
5. Parse amount: strip whitespace, remove leading "$", remove commas, convert to float; on failure skip row.
6. Only rows with status "paid" and valid amount and valid customer are aggregated; malformed rows (wrong field count, non-numeric amount, empty customer) skipped and counted. Non-"paid" status rows are valid but n


ITERATION 2 · model (THOUGHT / PLAN / ACTION)
THOUGHT: All hidden tests passed successfully.
ACTION: done

Model signalled completion.


## Final code + execution evidence

In [6]:
print(FINAL_CODE)
print()
ok, report = run_tests(FINAL_CODE)
print(report)
assert ok, "final code should pass all tests"

ns = {}; exec(FINAL_CODE, ns)
print("\nResult on SAMPLE:", json.dumps(ns["summarize_orders"](SAMPLE), indent=2))

import csv
import io

def summarize_orders(csv_text: str) -> dict:
    result = {"revenue_by_customer": {}, "total_revenue": 0.0, "skipped_rows": 0}

    if not csv_text or not csv_text.strip():
        return result

    f = io.StringIO(csv_text)
    reader = csv.reader(f)

    try:
        header = next(reader)
    except StopIteration:
        return result

    revenue_by_customer = {}
    total_revenue = 0.0
    skipped_rows = 0

    for row in reader:
        try:
            # skip completely empty rows (e.g. trailing blank lines)
            if len(row) == 1 and row[0].strip() == "":
                continue

            if len(row) != 4:
                skipped_rows += 1
                continue

            order_id, customer, amount_str, status = row

            customer = customer.strip() if customer is not None else ""
            status = status.strip().lower() if status is not None else ""
            amount_str = amount_str.strip() if amount_str is not None else ""

  

## Notes on fixes / iteration
- **Result:** the model passed all 6 tests on iteration 1, then correctly replied `ACTION: done` on iteration 2 after reading the observation. The PLAN shows it reasoned through the tricky part up front: it explicitly worked out that non-"paid" rows are valid and should *not* count toward `skipped_rows`, and chose `csv.reader` to handle quoted commas.
- **Harness validation:** before trusting a first-try pass, I checked the tests actually discriminate: a naive `line.split(",")` solution scores 3/6 (fails the quoted-name, thousands-separator and malformed-row tests), while a correct one scores 6/6.
- **Fixes to the loop itself:** (1) the response now includes a thinking block, so `resp.content[0].text` crashed — switched to joining text blocks; (2) with thinking enabled, one run spent the whole token budget before emitting code, so thinking is disabled and the loop relies on the explicit THOUGHT/PLAN stages in the prompt instead; (3) a test row was fixed to quote `"$1,250.00"` so the sample is valid CSV.